In [ ]:
import os.path
import pandas as pd

from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

In [ ]:

SCOPES = ["https://www.googleapis.com/auth/spreadsheets.readonly"]
SAMPLE_SPREADSHEET_ID = "1p-GqwHQpLoKJs-2u2yVdsDETPr8d_z7W2m9AOxPMS3E"
SAMPLE_RANGE_NAME = "sample!A:O"

def main() -> None:
  """
  Demonstrates how to use the Google Sheets API to read data from a spreadsheet.

  This function authenticates with the Google Sheets API using OAuth 2.0, then
  retrieves data from a specified spreadsheet and range.

  Args:
    None

  Returns:
    None

  Raises:
    HttpError: If an error occurs while communicating with the Google Sheets API.
  """
  creds = None

  if os.path.exists("token.json"):
    creds = Credentials.from_authorized_user_file("token.json", SCOPES)

  if not creds or not creds.valid:
    if creds and creds.expired and creds.refresh_token:
      creds.refresh(Request())
    else:
      flow = InstalledAppFlow.from_client_secrets_file(
          "../../config/credentials.json", SCOPES
      )
      creds = flow.run_local_server(port=0)

    with open("token.json", "w") as token:
      token.write(creds.to_json())

  try:
    service = build("sheets", "v4", credentials=creds)

    sheet = service.spreadsheets()
    result = (
        sheet.values()
        .get(spreadsheetId=SAMPLE_SPREADSHEET_ID, range=SAMPLE_RANGE_NAME)
        .execute()
    )
    values = result.get("values", [])

    if not values:
      print("No data found.")
      return
    
    for row in values:
      print(f"{row[0]}")

  except HttpError as err:
    print(err)

if __name__ == "__main__":
  main()